In [1]:
import pandas as pd
import numpy as np
import pickle

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedShuffleSplit, RandomizedSearchCV
from sklearn.metrics import (
    precision_score,
    recall_score,
    roc_auc_score,
    accuracy_score,
    f1_score
)
from imblearn.over_sampling import RandomOverSampler

In [2]:
seed = 42
n_splits = 5

# Stratified splits
kf = StratifiedShuffleSplit(
    n_splits=n_splits,
    test_size=1 / n_splits,
    random_state=seed
)

# Hyperparameter
param_grid = {
    "max_depth": [None, 3, 5, 10, 15, 20],
    "min_samples_split": [2, 5, 10, 20],
    "min_impurity_decrease": [0.0, 0.0001, 0.001, 0.01],
    "min_weight_fraction_leaf": [0.0, 0.01, 0.05, 0.1]
}

In [3]:
def evaluate_decision_tree(model_number, upsample=False):
    """
    Evaluate model using tuning, cross-validation and test data.
    """

    # Load training and test data
    X_train_full = pd.read_csv(
        f"../data/X_train_{model_number}.csv",
        keep_default_na=False
    )

    y_train_full = pd.read_csv(
        f"../data/y_train_{model_number}.csv",
        keep_default_na=False
    ).values.ravel()

    X_test = pd.read_csv(
        f"../data/X_test_{model_number}.csv",
        keep_default_na=False
    )

    y_test = pd.read_csv(
        f"../data/y_test_{model_number}.csv",
        keep_default_na=False
    ).values.ravel()

    # Check
    print("\n")
    print(model_number)

    print("X_train shape:", X_train_full.shape)
    print("X_test shape:", X_test.shape)

    print("\ny_train distribution:")
    print(pd.Series(y_train_full).value_counts())

    print("\ny_test distribution:")
    print(pd.Series(y_test).value_counts())

    if upsample:
        upsampler = RandomOverSampler(random_state=seed)
        X_tune, y_tune = upsampler.fit_resample(
            X_train_full,
            y_train_full
        )
    else:
        X_tune = X_train_full
        y_tune = y_train_full

    # Create a base Decision Tree model
    base_tree = DecisionTreeClassifier(
        random_state=seed
    )

    # Search for better hyperparameters
    random_search = RandomizedSearchCV(
        estimator=base_tree,
        param_distributions=param_grid,
        n_iter=30,
        scoring="roc_auc",
        cv=kf,
        random_state=seed,
        n_jobs=-1,
        verbose=1
    )
    random_search.fit(X_tune, y_tune)

    best_params = random_search.best_params_

    print("\nBest parameters:")
    print(best_params)

    print("\nBest tuning ROC AUC:")
    print(round(random_search.best_score_, 4))

    # Store cross-validation results
    cv_scores = {
        "fold": [],
        "cv_precision": [],
        "cv_recall": [],
        "cv_roc_auc": [],
        "cv_accuracy": [],
        "cv_f1": []
    }

    # Run cv using the best parameters
    splits = list(kf.split(X_train_full, y_train_full))

    for fold, (train_idx, val_idx) in enumerate(splits, start=1):

        # Split data into training and validation
        X_train = X_train_full.iloc[train_idx]
        y_train = y_train_full[train_idx]
        X_val = X_train_full.iloc[val_idx]
        y_val = y_train_full[val_idx]

        # Upsampling to the training
        if upsample:
            upsampler = RandomOverSampler(random_state=seed)
            X_train, y_train = upsampler.fit_resample(
                X_train,
                y_train
            )

        # Train model with the selected parameters
        clf = DecisionTreeClassifier(
            **best_params,
            random_state=seed
        )

        clf.fit(X_train, y_train)

        # Predict labels and probabilities
        preds = clf.predict(X_val)
        prob_preds = clf.predict_proba(X_val)[:, 1]

        # Save scores
        cv_scores["fold"].append(fold)
        cv_scores["cv_precision"].append(precision_score(y_val, preds, zero_division=0))
        cv_scores["cv_recall"].append(recall_score(y_val, preds, zero_division=0))
        cv_scores["cv_roc_auc"].append(roc_auc_score(y_val, prob_preds))
        cv_scores["cv_accuracy"].append(accuracy_score(y_val, preds))
        cv_scores["cv_f1"].append(f1_score(y_val, preds, zero_division=0))

    cv_results = pd.DataFrame(cv_scores)
    cv_summary = cv_results.drop(columns=["fold"]).agg(["mean", "sem"])

    print("\nCross-validation results:")
    display(cv_results)

    print("\nCross-validation summary:")
    display(cv_summary)

    if upsample:
        upsampler = RandomOverSampler(random_state=seed)
        X_train_final, y_train_final = upsampler.fit_resample(
            X_train_full,
            y_train_full
        )
    else:
        X_train_final = X_train_full
        y_train_final = y_train_full

    # Train the final model on the full training set
    final_model = DecisionTreeClassifier(
        **best_params,
        random_state=seed
    )

    final_model.fit(X_train_final, y_train_final)

    # Evaluate the final model on test set
    test_preds = final_model.predict(X_test)
    test_prob_preds = final_model.predict_proba(X_test)[:, 1]
    test_results = pd.DataFrame([{
        "test_precision": precision_score(y_test, test_preds, zero_division=0),
        "test_recall": recall_score(y_test, test_preds, zero_division=0),
        "test_roc_auc": roc_auc_score(y_test, test_prob_preds),
        "test_accuracy": accuracy_score(y_test, test_preds),
        "test_f1": f1_score(y_test, test_preds, zero_division=0)
    }])

    print("\nFinal test results:")
    display(test_results)

    return (
        best_params,
        cv_results,
        cv_summary,
        test_results,
        final_model
    )

In [4]:
# Run
model_settings = {
    "model_1": False,
    "model_1a": True,
    "model_1b": True,
    "model_2": False,
    "model_2a": True,
    "model_2b": True
}

best_params_dict = {}
cv_results_dict = {}
cv_summary_dict = {}
test_results_list = []
final_model_dict = {}

for model_name, use_upsample in model_settings.items():
    
    (
        best_params,
        cv_results,
        cv_summary,
        test_results,
        final_model
    ) = evaluate_decision_tree(
        model_name,
        upsample=use_upsample
    )

    best_params_dict[model_name] = best_params
    cv_results_dict[model_name] = cv_results
    cv_summary_dict[model_name] = cv_summary
    final_model_dict[model_name] = final_model

    test_results["model"] = model_name
    test_results_list.append(test_results)



model_1
X_train shape: (32108, 59)
X_test shape: (8028, 59)

y_train distribution:
1    17321
0    14787
Name: count, dtype: int64

y_test distribution:
1    4337
0    3691
Name: count, dtype: int64
Fitting 5 folds for each of 30 candidates, totalling 150 fits

Best parameters:
{'min_weight_fraction_leaf': 0.01, 'min_samples_split': 5, 'min_impurity_decrease': 0.0, 'max_depth': 15}

Best tuning ROC AUC:
0.8793

Cross-validation results:


,fold,cv_precision,cv_recall,cv_roc_auc,cv_accuracy,cv_f1
0,1,0.797065,0.815242,0.873983,0.788384,0.806051
1,2,0.809936,0.809469,0.878720,0.794768,0.809703
2,3,0.816565,0.808314,0.879845,0.798661,0.812418
3,4,0.807376,0.821594,0.878112,0.798038,0.814423
4,5,0.810034,0.829677,0.885671,0.803177,0.819738



Cross-validation summary:


,cv_precision,cv_recall,cv_roc_auc,cv_accuracy,cv_f1
mean,0.808195,0.816859,0.879266,0.796605,0.812466
sem,0.003170,0.003981,0.001883,0.002454,0.002297



Final test results:


...

,fold,cv_precision,cv_recall,cv_roc_auc,cv_accuracy,cv_f1
0,1,0.812838,0.823409,0.661930,0.734557,0.818089
1,2,0.806807,0.827515,0.652712,0.731332,0.817030
2,3,0.813126,0.805613,0.658893,0.724882,0.809352
3,4,0.810071,0.820329,0.656783,0.730340,0.815167
4,5,0.819449,0.804586,0.668749,0.729844,0.811950



Cross-validation summary:


,cv_precision,cv_recall,cv_roc_auc,cv_accuracy,cv_f1
mean,0.812458,0.816290,0.659813,0.730191,0.814318
sem,0.002087,0.004712,0.002691,0.001560,0.001621



Final test results:


,test_precision,test_recall,test_roc_auc,test_accuracy,test_f1
0,0.805987,0.801102,0.652148,0.717801,0.803537


In [5]:
# Summary
cv_results_all = pd.concat(cv_results_dict, names=["model", "row"])
cv_summary_all = pd.concat(cv_summary_dict, names=["model", "stat"])

test_summary_all = pd.concat(test_results_list, ignore_index=True)
test_summary_all = test_summary_all[
    [
        "model",
        "test_precision",
        "test_recall",
        "test_roc_auc",
        "test_accuracy",
        "test_f1"
    ]
]

best_params_all = pd.DataFrame(best_params_dict).T

print("Best parameters:")
display(best_params_all)

print("\nCV summaries:")
display(cv_summary_all)

print("\nTest results:")
display(test_summary_all)

best_params_all.to_csv("../data/decision_tree_best_params_all.csv")
cv_results_all.to_csv("../data/decision_tree_cv_results_all.csv")
cv_summary_all.to_csv("../data/decision_tree_cv_summary_all.csv")
test_summary_all.to_csv(
    "../data/decision_tree_test_summary_all.csv",
    index=False
)

with open("../data/decision_tree_final_models.pkl", "wb") as f:
    pickle.dump(final_model_dict, f)

Best parameters:


,min_weight_fraction_leaf,min_samples_split,min_impurity_decrease,max_depth
model_1,0.01,5.0,0.0,15.0
model_1a,0.00,20.0,0.0,20.0
model_1b,0.00,20.0,0.0,20.0
model_2,0.01,5.0,0.0,15.0
model_2a,0.01,5.0,0.0,15.0
model_2b,0.00,2.0,0.0,NaN



CV summaries:


cv_precision  cv_recall  cv_roc_auc  cv_accuracy     cv_f1
model    stat                                                            
model_1  mean      0.808195   0.816859    0.879266     0.796605  0.812466
         sem       0.003170   0.003981    0.001883     0.002454  0.002297
model_1a mean      0.499930   0.556169    0.728036     0.741639  0.526061
         sem       0.011855   0.004859    0.003628     0.006568  0.005074
model_1b mean      0.862179   0.815795    0.794668     0.777623  0.838334
         sem       0.001849   0.001842    0.002842     0.001356  0.000966
model_2  mean      0.769707   0.849080    0.784899     0.736032  0.807399
         sem       0.002824   0.004324    0.002367     0.002210  0.001671
model_2a mean      0.689209   0.688766    0.733087     0.671405  0.688974
         sem       0.003317   0.004235    0.005401     0.003528  0.003486
model_2b mean      0.812458   0.816290    0.659813     0.730191  0.814318
         sem       0.002087   0.004712    0.002691     0.001560  0.001621


Test results:


,model,test_precision,test_recall,test_roc_auc,test_accuracy,test_f1
0,model_1,0.806104,0.810007,0.875416,0.792103,0.808051
1,model_1a,0.496133,0.574904,0.723766,0.736367,0.532622
2,model_1b,0.857697,0.808493,0.796064,0.770192,0.832368
3,model_2,0.763421,0.862940,0.782663,0.736796,0.810136
4,model_2a,0.701077,0.694479,0.740225,0.679157,0.697762
5,model_2b,0.805987,0.801102,0.652148,0.717801,0.803537
